## Importações

In [ ]:
import collections
import gym
import numpy as np
import statistics
import tensorflow as tf
import tqdm

from tensorflow.keras import layers
from typing import Any, List, Sequence, Tuple

In [ ]:
# Create the environment
env = gym.make("CartPole-v1")

# Set seed for experiment reproducibility
seed = 42
env.reset(seed=seed)
tf.random.set_seed(seed)
np.random.seed(seed)

# Valor épsilon pequeno para estabilizar operações de divisão
eps = np.finfo(np.float32).eps.item()

/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


## Modelo
O Ator e o Crítico serão modelados usando uma rede neural que gera as probabilidades de ação e o valor crítico respectivamente. Este tutorial usa subclasses de modelo para definir o modelo.

Durante a passagem para frente, o modelo receberá o estado como entrada e produzirá as probabilidades de ação e o valor crítico
, que modela a função de valor dependente do estado . O objetivo é treinar um modelo que escolha ações com base em uma política
 que maximize o retorno esperado .

Para Cartpole-v0, existem quatro valores que representam o estado: posição do carrinho, velocidade do carrinho, ângulo do pólo e velocidade do pólo, respectivamente. O agente pode realizar duas ações para empurrar o carrinho para a esquerda (0) e para a direita (1), respectivamente.

Consulte a página wiki CartPole-v0 do OpenAI Gym para obter mais informações.

In [ ]:
class ActorCritic(tf.keras.Model):
  """Combined actor-critic network."""

  def __init__(self, num_actions: int, num_hidden_units: int):
    """Initialize."""
    super().__init__()

    self.common = layers.Dense(num_hidden_units, activation="relu")
    self.actor = layers.Dense(num_actions)
    self.critic = layers.Dense(1)

  def call(self, inputs: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    x = self.common(inputs)
    return self.actor(x), self.critic(x)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
num_actions = env.action_space.n  # 2
num_hidden_units = 128

model = ActorCritic(num_actions, num_hidden_units)

## Treinamento
Para treinar o agente, você seguirá estas etapas:

Execute o agente no ambiente para coletar dados de treinamento por episódio.        
Calcule o retorno esperado em cada passo de tempo.                
Calcule a perda para o modelo combinado ator-crítico.                     
Calcule gradientes e atualize os parâmetros de rede.
Repita 1-4 até o planejado de sucesso ou o máximo de episódios ocorridos.                
1. Coletando dados de treinamento                                           
Assim como no aprendizado supervisionado, para treinar o modelo ator-crítico, você precisa ter dados de treinamento. No entanto, para recolher esses dados, o modelo precisaria ser "executado" no ambiente.

Os dados de treinamento são coletados para cada episódio. Então, a cada passo de tempo, o avanço do modelo será executado no estado do ambiente para gerar probabilidades de ação e o valor crítico com base na política atual parametrizada pelos pesos do modelo.

A próxima ação será amostrada a partir das probabilidades de ação gerada pelo modelo, que então seria aplicada ao ambiente, fazendo com que o próximo estado e recompensa fossem gerados.

Esse processo é desenvolvido na função run_episode, que usa importações do TensorFlow para que possa ser compilado posteriormente em um gráfico do TensorFlow para um treinamento mais rápido. Observe que tf.TensorArrayforam usados ​​para dar suporte à iteração do Tensor em matrizes de comprimento variável.



In [ ]:
# Envolva a chamada `env.step` do OpenAI Gym como uma operação em uma função TensorFlow.
# Isso permitiria que ele fosse incluída em um gráfico do TensorFlow que pode ser chamado

def env_step(action: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
  """Returns state, reward and done flag given an action."""

  state, reward, done, _ = env.step(action)
  return (state.astype(np.float32),
          np.array(reward, np.int32),
          np.array(done, np.int32))

def tf_env_step(action: tf.Tensor) -> List[tf.Tensor]:
  return tf.numpy_function(env_step, [action],
                           [tf.float32, tf.int32, tf.int32])

In [ ]:
def run_episode(initial_state: tf.Tensor, model: tf.keras.Model, max_steps: int) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
    """Executa um único episódio para coletar dados de treinamento.

    Args:
        initial_state (tf.Tensor): Estado inicial do episódio.
        model (tf.keras.Model): Modelo para executar o episódio.
        max_steps (int): Número máximo de passos para o episódio.

    Returns:
        action_probs (tf.Tensor): Probabilidades de ação para cada passo.
        values (tf.Tensor): Valores críticos para cada passo.
        rewards (tf.Tensor): Recompensas para cada passo.
    """

    # Inicializa arrays dinâmicos para armazenar:
    action_probs = tf.TensorArray(dtype=tf.float32, size=0, dynamic_size=True)   # probabilidades de ação
    values = tf.TensorArray(dtype=tf.float32, size=0, dynamic_size=True)         # valores críticos
    rewards = tf.TensorArray(dtype=tf.int32, size=0, dynamic_size=True)          # recompensas

    # Obtém a forma do estado inicial para uso posterior
    initial_state_shape = initial_state.shape
    state = initial_state

    # Loop para cada passo no episódio, até o máximo de passos especificados
    for t in tf.range(max_steps):
        # Converte o estado atual em um tensor de tamanho de lote 1 (necessário pelo modelo)
        state = tf.expand_dims(state, 0)

        # Executa o modelo e obtém os logits de ação e o valor crítico
        # - `action_logits_t` são os logits das ações, que são pontuações não normalizadas para cada possível ação.
        # - `value` é o valor crítico, uma estimativa do valor esperado do estado atual
        action_logits_t, value = model(state)

        # Amostra a próxima ação a partir da distribuição de probabilidade de ação
        action = tf.random.categorical(action_logits_t, 1)[0, 0]
        action_probs_t = tf.nn.softmax(action_logits_t)

        # Armazena o valor crítico atual
        values = values.write(t, tf.squeeze(value))

        # Armazena a probabilidade da ação escolhida
        action_probs = action_probs.write(t, action_probs_t[0, action])

        # Aplica a ação ao ambiente para obter o próximo estado e recompensa
        state, reward, done = tf_env_step(action)
        state.set_shape(initial_state_shape)

        # Armazena a recompensa atual
        rewards = rewards.write(t, reward)

        # Se o episódio estiver concluído, saia do loop
        if tf.cast(done, tf.bool):
            break

    # Converte os arrays dinâmicos em tensores
    action_probs = action_probs.stack()
    values = values.stack()
    rewards = rewards.stack()

    return action_probs, values, rewards

## Calculando os retornos esperados

In [ ]:
def get_expected_return(
    rewards: tf.Tensor,
    gamma: float,
    standardize: bool = True) -> tf.Tensor:
    """Calcule os retornos esperados por intervalo de tempo
    Args:
      rewards (tf.Tensor): Recompensas para cada passo.
      gamma (float): Taxa de desconto.
      standardize (bool): Padronize os retornos.

    Returns:
      returns (tf.Tensor): Retornos esperados para cada passo.
    """

    # Obtém o número de recompensas
    n = tf.shape(rewards)[0]

    # Inicializa um TensorArray para armazenar os retornos
    returns = tf.TensorArray(dtype=tf.float32, size=n)

    # Inverte a ordem das recompensas e converte para float32
    # Fazendo isso, podemos acumular a soma descontada de trás para frente
    rewards = tf.cast(rewards[::-1], dtype=tf.float32)

    # Inicializa a soma descontada
    discounted_sum = tf.constant(0.0)
    discounted_sum_shape = discounted_sum.shape

    # Loop para calcular a soma descontada das recompensas
    for i in tf.range(n):
      reward = rewards[i] # Obtém a recompensa na posição i
       # Atualiza a soma descontada adicionando a recompensa atual
       # e multiplicando a soma descontada anterior por gamma (fator de desconto)
      discounted_sum = reward + gamma * discounted_sum
      discounted_sum.set_shape(discounted_sum_shape) # Mantém a forma original da soma descontada
      returns = returns.write(i, discounted_sum)     # Armazena a soma descontada atual no TensorArray 'returns' na posição i

    # Inverte novamente a ordem dos retornos para corresponder à ordem original das recompensas
    returns = returns.stack()[::-1]

    # Se a padronização for verdadeira, centraliza e escala os retornos
    if standardize:
        returns = ((returns - tf.math.reduce_mean(returns)) /
                   (tf.math.reduce_std(returns) + tf.keras.backend.epsilon()))

    return returns

## Loss

In [ ]:
# Define a função de perda Huber com a redução de soma
huber_loss = tf.keras.losses.Huber(reduction=tf.keras.losses.Reduction.SUM)

def compute_loss(
    action_probs: tf.Tensor,
    values: tf.Tensor,
    returns: tf.Tensor) -> tf.Tensor:
    """Computes the combined actor-critic loss.
    Args:
      action_probs (tf.Tensor): Probabilidades de ação para cada passo.
      values (tf.Tensor): Valores críticos para cada passo.
      returns (tf.Tensor): Retornos esperados para cada passo.
    Returns:
      loss (tf.Tensor): Perda do modelo combinado ator-crítico.
    """

    # Calcula a vantagem como a diferença entre os retornos esperados
    # e os valores críticos estimados
    advantage = returns - values

    # Calcula o logaritmo das probabilidades das ações
    # Isso é usado para estabilizar os cálculos e facilitar a derivação dos gradientes
    action_log_probs = tf.math.log(action_probs)

    # Calcula a perda do ator
    # Multiplica o log das probabilidades das ações pela vantagem e soma (reduz) os valores resultantes
    actor_loss = -tf.math.reduce_sum(action_log_probs * advantage)

    # Calcula a perda do crítico usando a função de perda Huber
    # Compara os valores críticos estimados com os retornos esperados
    critic_loss = huber_loss(values, returns)

    return actor_loss + critic_loss # Retorna a soma das perdas do ator e do crítico

## Etapa de treinamento para atualizar os parâmetros - Otimizador

In [ ]:
# Define o otimizador Adam com uma taxa de aprendizado de 0.01
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

@tf.function
def train_step(initial_state: tf.Tensor, model: tf.keras.Model,
               optimizer: tf.keras.optimizers.Optimizer, gamma: float,
               max_steps_per_episode: int) -> tf.Tensor:
    """Executa um passo de treinamento do modelo
    Args:
      initial_state (tf.Tensor): Estado inicial do episódio.
      model (tf.keras.Model): Modelo para executar o episódio.
      optimizer (tf.keras.optimizers.Optimizer): Otimizador para atualizar os parâmetros do modelo.
      gamma (float): Taxa de desconto.
      max_steps_per_episode (int): Número máximo de passos para o episódio.
    Returns:
      episode_reward (tf.Tensor): Recompensa total do episódio.
    """

    # Abre um contexto de gravação do GradientTape para rastrear operações para diferenciação automática
    with tf.GradientTape() as tape:

        # Executa o modelo para um episódio para coletar dados de treinamento
        action_probs, values, rewards = run_episode(
            initial_state, model, max_steps_per_episode)

        # Calcula os retornos esperados
        returns = get_expected_return(rewards, gamma)

        # Converte os dados de treinamento para formas apropriadas de tensores   TF
        # Cada tensor (action_probs, values, returns) é expandido para adicionar uma dimensão extra
        action_probs, values, returns = [
            tf.expand_dims(x, 1) for x in [action_probs, values, returns]]

        # Calcula os valores de perda para atualizar a nossa rede
        loss = compute_loss(action_probs, values, returns)

    # Calcula os gradientes a partir da perda
    grads = tape.gradient(loss, model.trainable_variables)

    # Aplica os gradientes aos parâmetros do modelo
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    # Calcula a recompensa total do episódio
    episode_reward = tf.math.reduce_sum(rewards)

    return episode_reward

## Loop de treinamento

In [2]:
# Medir o tempo de execução do código
%%time

# Definindo critérios e parâmetros de treinamento
min_episodes_criterion = 100  # Número mínimo de episódios para considerar a resolução
max_episodes = 10000          # Número máximo de episódios para treinar
max_steps_per_episode = 1000  # Número máximo de passos por episódio

# O ambiente Cartpole-v0 é considerado resolvido se a recompensa média for >= 195
# em 100 tentativas consecutivas

reward_threshold = 195  # Limite de recompensa para considerar o problema resolvido
running_reward = 0      # Recompensa média atual

gamma = 0.99  # Taxa de desconto

# Guardar as recompensas dos últimos episódios
episodes_reward: collections.deque = collections.deque(maxlen=min_episodes_criterion)

with tqdm.trange(max_episodes) as t:
  for i in t:
    # Obter o estado inicial do ambiente
    initial_state = tf.constant(env.reset(), dtype=tf.float32)

    # Executar um passo de treinamento e obter a recompensa do episódio
    episode_reward = int(train_step(
        initial_state, model, optimizer, gamma, max_steps_per_episode))

    episodes_reward.append(episode_reward)            # Adicionar a recompensa do episódio à lista de recompensas
    running_reward = statistics.mean(episodes_reward) # Calcular a recompensa média corrente

    # Atualizar a descrição da barra de progresso
    t.set_description(f'Episode {i}')
    t.set_postfix(
        episode_reward=episode_reward, running_reward=running_reward)

    # Mostrar a recompensa média do episódio a cada 10 episódios
    if i % 10 == 0:
      pass # print(f'Episode {i}: average reward: {avg_reward}')

    # Verificar se a recompensa média atinge o limiar e se o número mínimo de episódios foi alcançado
    if running_reward > reward_threshold and i >= min_episodes_criterion:
        break

# Exibir mensagem de resolução do problema
print(f'\nResolvido no episódio {i}: recompensa média: {running_reward:.2f}!')

NameError: name 'collections' is not defined